# 01 — Ingestion and Raw Validation

**Tujuan:** mengambil dataset mentah, melakukan sanity check/validasi struktur dasar, mencatat metadata ingestion, lalu menyimpan hasil ke `data/staged/` dalam format Parquet.

Notebook ini **tidak melakukan business transformation**. Normalisasi nama kolom ke lowercase diperlakukan sebagai standardisasi schema teknis.

## 1. Import dan konfigurasi

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import polars as pl

TRAIN_RAW = Path("../data/raw/train.csv")
TEST_RAW = Path("../data/raw/test.csv")
STAGED_DIR = Path("../data/staged")

STAGED_DIR.mkdir(parents=True, exist_ok=True)

print(f"TRAIN_RAW : {TRAIN_RAW}")
print(f"TEST_RAW  : {TEST_RAW}")
print(f"OUTPUT    : {STAGED_DIR}")

TRAIN_RAW : ..\data\raw\train.csv
TEST_RAW  : ..\data\raw\test.csv
OUTPUT    : ..\data\staged


## 2. Definisi schema yang diharapkan

Schema ditulis lowercase agar konsisten dengan dataset staged.

In [2]:
EXPECTED_TRAIN_COLUMNS = [
    "passengerid", "survived", "pclass", "name", "sex",
    "age", "sibsp", "parch", "ticket", "fare", "cabin", "embarked"
]

EXPECTED_TEST_COLUMNS = [
    "passengerid", "pclass", "name", "sex",
    "age", "sibsp", "parch", "ticket", "fare", "cabin", "embarked"
]

EXPECTED_DTYPES = {
    "passengerid": pl.Int64,
    "survived": pl.Int64,
    "pclass": pl.Int64,
    "name": pl.String,
    "sex": pl.String,
    "age": pl.Float64,
    "sibsp": pl.Int64,
    "parch": pl.Int64,
    "ticket": pl.String,
    "fare": pl.Float64,
    "cabin": pl.String,
    "embarked": pl.String,
}

## 3. File sanity check

Memastikan file tersedia, bukan direktori, tidak kosong, dan dapat dibaca sebagai CSV.

In [3]:
def validate_file(path: Path) -> dict:
    assert path.exists(), f"File tidak ditemukan: {path}"
    assert path.is_file(), f"Path bukan file: {path}"
    size = path.stat().st_size
    assert size > 0, f"File kosong: {path}"

    return {
        "path": str(path),
        "size_bytes": size,
    }

train_file_info = validate_file(TRAIN_RAW)
test_file_info = validate_file(TEST_RAW)

print(train_file_info)
print(test_file_info)

{'path': '..\\data\\raw\\train.csv', 'size_bytes': 61194}
{'path': '..\\data\\raw\\test.csv', 'size_bytes': 28629}


## 4. Hash file untuk identifikasi versi source

In [4]:
def sha256_file(path: Path) -> str:
    sha256 = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)
    return sha256.hexdigest()

train_sha256 = sha256_file(TRAIN_RAW)
test_sha256 = sha256_file(TEST_RAW)

print("train sha256:", train_sha256)
print("test  sha256:", test_sha256)

train sha256: 7d118fef8b6ccf7f81111877bc388536f7b1e498a655e3d649d19aaa010e9f6f
test  sha256: 56023b9948236f3c7a1c9448fcf418b283e109ef177fa8c7e069158dd7dd52b2


## 5. Load raw CSV

Missing-value tokens dinormalisasi menjadi null. Selain standardisasi schema, tidak dilakukan transformasi bisnis.

In [ ]:
MISSING_VALUES = ["N/a", "n/a", "No", r"N**\a**", r"N\a", "na", "NA", ""]

def load_raw_csv(path: Path) -> pl.DataFrame:
    return pl.read_csv(
        path,
        null_values=MISSING_VALUES,
        try_parse_dates=False,
    )

try:
    train = load_raw_csv(TRAIN_RAW)
    test = load_raw_csv(TEST_RAW)

except Exception as exc:
    raise RuntimeError(f"Gagal membaca CSV. File mungkin corrupt / tidak valid: {exc}") from exc

print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (891, 12)
Test shape : (418, 11)


## 6. Standardisasi nama kolom

Hanya nama kolom yang dinormalisasi menjadi lowercase. Nilai dan struktur record tidak diubah.

In [6]:
def normalize_column_names(df: pl.DataFrame) -> pl.DataFrame:
    return df.rename({col: col.lower() for col in df.columns})

train = normalize_column_names(train)
test = normalize_column_names(test)

print("Train columns:", train.columns)
print("Test columns :", test.columns)

Train columns: ['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']
Test columns : ['passengerid', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked']


## 7. Validasi jumlah kolom dan nama kolom

In [7]:
assert train.width == len(EXPECTED_TRAIN_COLUMNS), (
    f"Jumlah kolom train salah: {train.width}"
)
assert test.width == len(EXPECTED_TEST_COLUMNS), (
    f"Jumlah kolom test salah: {test.width}"
)

assert train.columns == EXPECTED_TRAIN_COLUMNS, (
    f"Train columns tidak sesuai: {train.columns}"
)
assert test.columns == EXPECTED_TEST_COLUMNS, (
    f"Test columns tidak sesuai: {test.columns}"
)

print("✓ Jumlah dan nama kolom valid")

✓ Jumlah dan nama kolom valid


## 8. Validasi tipe data dasar

In [8]:
def validate_dtypes(df: pl.DataFrame, dataset_name: str):
    for column, expected_dtype in EXPECTED_DTYPES.items():
        if column not in df.columns:
            continue
        actual_dtype = df.schema[column]
        assert actual_dtype == expected_dtype, (
            f"{dataset_name}.{column}: expected {expected_dtype}, got {actual_dtype}"
        )

validate_dtypes(train, "train")
validate_dtypes(test, "test")

print("✓ Tipe data dasar valid")
print("Train schema:")
print(train.schema)

✓ Tipe data dasar valid
Train schema:
Schema({'passengerid': Int64, 'survived': Int64, 'pclass': Int64, 'name': String, 'sex': String, 'age': Float64, 'sibsp': Int64, 'parch': Int64, 'ticket': String, 'fare': Float64, 'cabin': String, 'embarked': String})


## 9. Validasi jumlah baris dan dataset kosong

In [9]:
assert train.height > 0, "Train dataset kosong"
assert test.height > 0, "Test dataset kosong"

print(f"✓ Train rows: {train.height:,}")
print(f"✓ Test rows : {test.height:,}")

✓ Train rows: 891
✓ Test rows : 418


## 10. Missing-value report

Tahap ini hanya melaporkan missing value; belum melakukan imputasi.

In [10]:
def missing_report(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.null_count()
        .transpose(
            include_header=True,
            header_name="column",
            column_names=["null_count"],
        )
        .with_columns(
            (pl.col("null_count") / df.height * 100).round(2).alias("null_percent")
        )
    )

print("TRAIN")
display(missing_report(train))

print("TEST")
display(missing_report(test))

TRAIN


column,null_count,null_percent
str,u32,f64
"""passengerid""",0,0.0
"""survived""",0,0.0
"""pclass""",0,0.0
"""name""",0,0.0
"""sex""",0,0.0
…,…,…
"""parch""",0,0.0
"""ticket""",0,0.0
"""fare""",0,0.0


TEST


column,null_count,null_percent
str,u32,f64
"""passengerid""",0,0.0
"""pclass""",0,0.0
"""name""",0,0.0
"""sex""",0,0.0
"""age""",86,20.57
…,…,…
"""parch""",0,0.0
"""ticket""",0,0.0
"""fare""",1,0.24


## 11. Basic value sanity check

Pengecekan ini hanya memastikan nilai-nilai kunci tidak berada di luar domain dasar dataset. Tidak ada feature engineering.

In [11]:
assert train["passengerid"].n_unique() == train.height, (
    "passengerid train mengandung duplicate"
)

assert train["survived"].drop_nulls().is_in([0, 1]).all(), (
    "survived hanya boleh bernilai 0 atau 1"
)

assert train["pclass"].drop_nulls().is_in([1, 2, 3]).all(), (
    "pclass hanya boleh bernilai 1, 2, atau 3"
)

assert test["passengerid"].n_unique() == test.height, (
    "passengerid test mengandung duplicate"
)

assert test["pclass"].drop_nulls().is_in([1, 2, 3]).all(), (
    "test.pclass hanya boleh bernilai 1, 2, atau 3"
)

print("✓ Basic value sanity check lulus")

✓ Basic value sanity check lulus


## 12. Ringkasan raw dataset

In [12]:
print("TRAIN")
display(train.head())
display(train.describe())

print("TEST")
display(test.head())
display(test.describe())

TRAIN


passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,i64,str,str,f64,i64,i64,str,f64,str,str
1,0,3,"""Braund, Mr. Owen Harris""","""male""",22.0,1,0,"""A/5 21171""",7.25,null,"""S"""
2,1,1,"""Cumings, Mrs. John Bradley (Fl…","""female""",38.0,1,0,"""PC 17599""",71.2833,"""C85""","""C"""
3,1,3,"""Heikkinen, Miss. Laina""","""female""",26.0,0,0,"""STON/O2. 3101282""",7.925,null,"""S"""
4,1,1,"""Futrelle, Mrs. Jacques Heath (…","""female""",35.0,1,0,"""113803""",53.1,"""C123""","""S"""
5,0,3,"""Allen, Mr. William Henry""","""male""",35.0,0,0,"""373450""",8.05,null,"""S"""


statistic,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",891.0,891.0,891.0,"""891""","""891""",714.0,891.0,891.0,"""891""",891.0,"""204""","""889"""
"""null_count""",0.0,0.0,0.0,"""0""","""0""",177.0,0.0,0.0,"""0""",0.0,"""687""","""2"""
"""mean""",446.0,0.383838,2.308642,null,null,29.699118,0.523008,0.381594,null,32.204208,null,null
"""std""",257.353842,0.486592,0.836071,null,null,14.526497,1.102743,0.806057,null,49.693429,null,null
"""min""",1.0,0.0,1.0,"""Abbing, Mr. Anthony""","""female""",0.42,0.0,0.0,"""110152""",0.0,"""A10""","""C"""
"""25%""",224.0,0.0,2.0,null,null,20.0,0.0,0.0,null,7.925,null,null
"""50%""",446.0,0.0,3.0,null,null,28.0,0.0,0.0,null,14.4542,null,null
"""75%""",669.0,1.0,3.0,null,null,38.0,1.0,0.0,null,31.0,null,null
"""max""",891.0,1.0,3.0,"""van Melkebeke, Mr. Philemon""","""male""",80.0,8.0,6.0,"""WE/P 5735""",512.3292,"""T""","""S"""


TEST


passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
i64,i64,str,str,f64,i64,i64,str,f64,str,str
892,3,"""Kelly, Mr. James""","""male""",34.5,0,0,"""330911""",7.8292,null,"""Q"""
893,3,"""Wilkes, Mrs. James (Ellen Need…","""female""",47.0,1,0,"""363272""",7.0,null,"""S"""
894,2,"""Myles, Mr. Thomas Francis""","""male""",62.0,0,0,"""240276""",9.6875,null,"""Q"""
895,3,"""Wirz, Mr. Albert""","""male""",27.0,0,0,"""315154""",8.6625,null,"""S"""
896,3,"""Hirvonen, Mrs. Alexander (Helg…","""female""",22.0,1,1,"""3101298""",12.2875,null,"""S"""


statistic,passengerid,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked
str,f64,f64,str,str,f64,f64,f64,str,f64,str,str
"""count""",418.0,418.0,"""418""","""418""",332.0,418.0,418.0,"""418""",417.0,"""91""","""418"""
"""null_count""",0.0,0.0,"""0""","""0""",86.0,0.0,0.0,"""0""",1.0,"""327""","""0"""
"""mean""",1100.5,2.26555,null,null,30.27259,0.447368,0.392344,null,35.627188,null,null
"""std""",120.810458,0.841838,null,null,14.181209,0.89676,0.981429,null,55.907576,null,null
"""min""",892.0,1.0,"""Abbott, Master. Eugene Joseph""","""female""",0.17,0.0,0.0,"""110469""",0.0,"""A11""","""C"""
"""25%""",996.0,1.0,null,null,21.0,0.0,0.0,null,7.8958,null,null
"""50%""",1101.0,3.0,null,null,27.0,0.0,0.0,null,14.4542,null,null
"""75%""",1205.0,3.0,null,null,39.0,1.0,0.0,null,31.5,null,null
"""max""",1309.0,3.0,"""van Billiard, Master. Walter J…","""male""",76.0,8.0,9.0,"""W.E.P. 5734""",512.3292,"""G6""","""S"""


## 13. Simpan raw staged ke Parquet

Output ini merupakan hasil ingestion setelah sanity check dan standardisasi nama kolom.

In [13]:
TRAIN_STAGED = STAGED_DIR / "train.parquet"
TEST_STAGED = STAGED_DIR / "test.parquet"

train.write_parquet(TRAIN_STAGED)
test.write_parquet(TEST_STAGED)

assert TRAIN_STAGED.exists() and TRAIN_STAGED.stat().st_size > 0
assert TEST_STAGED.exists() and TEST_STAGED.stat().st_size > 0

print(f"✓ Written: {TRAIN_STAGED}")
print(f"✓ Written: {TEST_STAGED}")

✓ Written: ..\data\staged\train.parquet
✓ Written: ..\data\staged\test.parquet


## 14. Metadata ingestion

In [14]:
ingestion_timestamp = datetime.now(timezone.utc).isoformat()

metadata = {
    "dataset": "titanic",
    "source_type": "local_csv",
    "source_version": {
        "train_sha256": train_sha256,
        "test_sha256": test_sha256,
    },
    "ingestion_timestamp_utc": ingestion_timestamp,
    "train": {
        "source_file": str(TRAIN_RAW),
        "staged_file": str(TRAIN_STAGED),
        "rows": train.height,
        "columns": train.width,
        "size_bytes_raw": TRAIN_RAW.stat().st_size,
        "size_bytes_staged": TRAIN_STAGED.stat().st_size,
        "schema": {k: str(v) for k, v in train.schema.items()},
    },
    "test": {
        "source_file": str(TEST_RAW),
        "staged_file": str(TEST_STAGED),
        "rows": test.height,
        "columns": test.width,
        "size_bytes_raw": TEST_RAW.stat().st_size,
        "size_bytes_staged": TEST_STAGED.stat().st_size,
        "schema": {k: str(v) for k, v in test.schema.items()},
    },
}

METADATA_PATH = STAGED_DIR / "ingestion_metadata.json"
with METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✓ Metadata written: {METADATA_PATH}")

✓ Metadata written: ..\data\staged\ingestion_metadata.json


## 15. Final validation terhadap artifact staged

Baca kembali Parquet untuk memastikan artifact yang ditulis dapat dibuka dan schema/row count tetap konsisten.

In [15]:
train_staged_check = pl.read_parquet(TRAIN_STAGED)
test_staged_check = pl.read_parquet(TEST_STAGED)

assert train_staged_check.columns == EXPECTED_TRAIN_COLUMNS
assert test_staged_check.columns == EXPECTED_TEST_COLUMNS
assert train_staged_check.height == train.height
assert test_staged_check.height == test.height

print("✓ Staged artifact dapat dibaca kembali")
print("✓ Schema konsisten")
print("✓ Row count konsisten")

✓ Staged artifact dapat dibaca kembali
✓ Schema konsisten
✓ Row count konsisten


## 16. Final ingestion report

Jika cell ini berhasil, ingestion layer selesai dan artifact siap digunakan oleh notebook tahap berikutnya.

In [16]:
print("=" * 60)
print("INGESTION COMPLETED")
print("=" * 60)
print(f"Dataset       : {metadata['dataset']}")
print(f"Source        : {metadata['source_type']}")
print(f"Timestamp UTC : {metadata['ingestion_timestamp_utc']}")
print(f"Train         : {train.height:,} rows x {train.width} cols")
print(f"Test          : {test.height:,} rows x {test.width} cols")
print(f"Output train  : {TRAIN_STAGED}")
print(f"Output test   : {TEST_STAGED}")
print(f"Metadata      : {METADATA_PATH}")
print("Status        : PASS")

INGESTION COMPLETED
Dataset       : titanic
Source        : local_csv
Timestamp UTC : 2026-09-16T08:30:04.275927+00:00
Train         : 891 rows x 12 cols
Test          : 418 rows x 11 cols
Output train  : ..\data\staged\train.parquet
Output test   : ..\data\staged\test.parquet
Metadata      : ..\data\staged\ingestion_metadata.json
Status        : PASS
